 
**Dashboard URL:** https://hivcaregapai.streamlit.app/

---

## **Purpose of This Notebook**

This notebook is a **deployment handover document**; not executable code.  
It explains how the HIV Care Gap AI dashboard was deployed to Streamlit Cloud, what decisions were made during deployment, what problems were encountered and how they were resolved, and how to redeploy or update the system in future years.

Anyone reading this notebook; including a future MOH data team, a new developer,or a technical reviewer, should be able to take the GitHub repository and have the dashboard live on Streamlit Cloud within 10 minutes.

---
## **Section 1: What Was Deployed and Where**

The HIV Care Gap AI system was deployed as a public Streamlit web application hosted on Streamlit Cloud (free tier).

### **Live URLs**

| | URL |
|-|-----|
| **Primary (renamed)** | https://hivcaregapai.streamlit.app/ |
| **Original auto-generated** | https://pj2egzlzpl4jugcvnkd6qv.streamlit.app/ |

The URL was renamed from the auto-generated Streamlit subdomain to `hivcaregapai` using the **Rename app** option in the Streamlit Cloud dashboard settings.  
Both URLs resolve to the same deployment.

### **What the dashboard contains**

The deployed app is `app/streamlit_app.py`; a three-tab Streamlit dashboard:

| Tab | Title | What it shows |
|-----|-------|---------------|
| Tab 1 | County Tier Map | Ranked bar chart of all 47 counties by Care Gap Index, Folium choropleth map, tier metrics |
| Tab 2 | Dropout Risk Factors | Odds ratio forest plot with 95% CI, logistic regression performance metrics, demographic profile explorer |
| Tab 3 | Scenario Projections | Dual-scenario line charts (BAU vs Bridged Gap) per tier and nationally, patients-retained counter, county comparison table, CSV export |

### **Platform details**

| Item | Detail |
|------|--------|
| Platform | Streamlit Cloud (share.streamlit.io) |
| Tier | Free |
| Python version | 3.8.5 |
| Main file | `app/streamlit_app.py` |
| Branch | `main` |
| Auto-redeploy | Yes - every push to `main` triggers a redeploy |

---
## Section 2: Repository Structure for Deployment

Streamlit Cloud clones the GitHub repository and installs packages from `requirements.txt`.  
It does **not** have access to any files on your local computer.  
Everything the app needs to run must be committed to the repository.

The files that must be present in the repo for the app to function:

```
requirements.txt                       ← Streamlit Cloud installs these
constants.py                           ← All file paths and model parameters
app/
    streamlit_app.py                   ← Main app entry point
src/
    projection.py                      ← Used by Tab 3
    feature_engineering.py             ← Used by Tab 1
    nsdcc_cleaner.py                   ← Utility functions
    dhs_cleaner.py                     ← Utility functions
    model_training.py                  ← Utility functions
    evaluation.py                      ← Utility functions
data/
    processed/
        county_profiles.csv            ← Tab 1 data source
        nsdcc_clean.csv                ← Merged NSDCC data
        hts_clean.csv                  ← Cleaned HTS
        hts_positive_clean.csv         ← Cleaned HTS Positive
        art_clean.csv                  ← Cleaned ART
        vlt_clean.csv                  ← Cleaned VLT
        iit_clean.csv                  ← Cleaned IIT
        individual_features_clean.csv  ← Cleaned DHS data
        odds_ratios_with_ci.json       ← Tab 2 forest plot data
        logreg_baseline.json           ← Tab 2 performance metrics
        dropout_risk_factors.csv       ← Tab 2 fallback data
        forecast_critical.csv          ← Tab 3 Critical tier projections
        forecast_high.csv              ← Tab 3 High tier projections
        forecast_moderate.csv          ← Tab 3 Moderate tier projections
        forecast_low.csv               ← Tab 3 Low tier projections
        forecast_national.csv          ← Tab 3 national projections
        patients_retained.csv          ← Tab 3 patients saved counter
        county_comparison.csv          ← Tab 3 comparison table
        iit_alerts.csv                 ← Alert counties
models/
    xgboost_dropout.pkl                ← Logistic regression model bundle
    kmeans_county_tiers.pkl            ← KMeans model bundle
    model3_scenario.pkl                ← Model 3 projection bundle
```

> ### **Important note on naming:**
>`models/xgboost_dropout.pkl` contains the **logistic regression bundle**, not an XGBoost model.  
The file is named `xgboost_dropout.pkl` for Streamlit compatibility because the path constant `XGBOOST_MODEL` was already defined in `constants.py` and referenced throughout the codebase.  
The bundle contains: the fitted `LogisticRegression` model, label encoders, feature names, performance metrics, odds ratios, and CI bounds. This is documented in `scripts/train_model2.py`.

---
## **Section 3: What Was Committed to GitHub**

### **The .gitignore situation:**

- By default, `.pkl` files and `data/processed/` are excluded from Git because they are generated outputs — large binary files that should be regenerated by running the pipeline, not stored in version control.

- However, Streamlit Cloud cannot run the pipeline on startup — it only serves the app.  
This means all data files and model bundles **must** be committed to the repository for the deployment to work.

### **What was added to the repository for deployment:**

The following `.gitignore` exceptions were added to allow these files to be committed:

```
In .gitignore — add these exception lines:
!data/processed/*.csv
!data/processed/*.json
!models/*.pkl
```

Then all processed data files and model bundles were committed:

```bash
git add data/processed/
git add models/
git commit -m "feat: Add processed data and model bundles for Streamlit Cloud deployment"
git push origin main
```

### **Raw data files**

- The raw Excel files in `data/raw/` (ART, HTS, HTS_Positive, VLT, IIT) and  
`data/raw/individual_features.csv` (DHS data) are **not** committed to the repository.  
- They are downloaded directly from source:

| Dataset | Source | Requires login? |
|---------|--------|-----------------|
| NSDCC raw files (ART, HTS, VLT, IIT, HTS_Positive) | analytics.nsdcc.go.ke/estimates | No |
| DHS Kenya 2022 Individual Recode | dhsprogram.com/data/dataset/Kenya_Standard-DHS_2022.cfm | Yes (approved) |

These are only needed if you are regenerating the pipeline from scratch.

---
## **Section 4: How Streamlit Cloud Deployment Works**

### **Step-by-step: how to deploy from a fresh GitHub repository**

**Step 1: Ensure all required files are committed**

Run the full pipeline locally first:
```bash
python main.py
```
This generates all files in `data/processed/` and `models/`.  
Confirm nothing is missing:
```bash
python main.py --dry-run
```
Then commit everything needed for deployment:
```bash
git add data/processed/ models/ requirements.txt constants.py
git commit -m "Deployment: add all processed outputs and model bundles"
git push origin main
```

**Step 2: Log in to Streamlit Cloud**

Go to https://share.streamlit.io and sign in with your GitHub account.  
Streamlit Cloud needs read access to the repository.

**Step 3: Create a new app**

Click **New app** and fill in the form:

| Field | Value |
|-------|-------|
| Repository | `<your-github-username>/<repo-name>` |
| Branch | `main` |
| Main file path | `app/streamlit_app.py` |
| App URL (optional) | `hivcaregapai` |

Click **Deploy**.

**Step 4: Wait for build to complete**

Streamlit Cloud will:
1. Clone the repository
2. Install all packages from `requirements.txt`
3. Launch `app/streamlit_app.py`

Build typically takes 2–4 minutes on first deploy.  
Subsequent deploys (triggered by `git push`) take about 1–2 minutes.

**Step 5: Rename the app URL (optional)**

After deployment, go to the app settings in the Streamlit Cloud dashboard.  
Under **General**, find the **URL** field and change it from the auto-generated string to a readable name such as hivcaregapai`.  
The app will then be accessible at `https://hivcaregapai.streamlit.app/`.

### **How auto-redeploy works:**

Every time you push to the `main` branch on GitHub, Streamlit Cloud automatically redeploys the app. You do not need to do anything in the Streamlit Cloud dashboard.  
The app will show a "rerunning" spinner for about 1–2 minutes during redeployment.

---
## **Section 5: Problems Encountered and How They Were Resolved**

### **Problem 1: Model file not found: `xgboost_dropout.pkl`**

**What happened:**  
The Demographic Profile Explorer in Tab 2 showed the warning:

> ⚠ Logistic Regression model not found. Run `scripts/train_model2.py` to generate it.

The dashboard was looking for `models/xgboost_dropout.pkl` (the path defined in `constants.py` as `XGBOOST_MODEL`) but the file was not in the GitHub repository.  
Streamlit Cloud only has access to files that are committed to Git - it cannot see files on your local computer.

**Why it happened:**  
The `.gitignore` file contained `*.pkl`, which prevented model files from being committed.  
This is correct practice for large binary files in general, but it means Streamlit Cloud cannot access the trained model.

**How it was resolved:**  
The exception line `!models/*.pkl` was added to `.gitignore` to allow pkl files to be committed.  
Then `models/xgboost_dropout.pkl` was added to the repository:

```bash
git add models/xgboost_dropout.pkl
git commit -m "Add trained model file for Streamlit Cloud deployment"
git push
```

After the push, Streamlit Cloud redeployed automatically and the model loaded correctly.

**Important note on what this file contains:**  
Despite the filename `xgboost_dropout.pkl`, this file contains the **logistic regression bundle** produced by `scripts/train_model2.py`. XGBoost was dropped from the project due to extreme class imbalance (26 dropout cases out of 32,156 records). The filename was preserved for compatibility with the existing constant `XGBOOST_MODEL` in `constants.py`.  
The bundle contains: fitted `LogisticRegression` model, label encoders, feature names, performance metrics, and odds ratio data.

---

### **Problem 2: App URL was an unreadable auto-generated string**

**What happened:**  
The initial deployment URL was `https://pj2egzlzpl4jugcvnkd6qv.streamlit.app/` - an auto-generated string with no meaning.

**How it was resolved:**  
The URL was renamed to `hivcaregapai` using the **Rename app** option in the Streamlit Cloud app settings panel.  
The app is now accessible at: **https://hivcaregapai.streamlit.app/**

---

### **Problem 3: Folium map requires internet on Streamlit Cloud**

**What happened:**  
The Folium choropleth map in Tab 1 uses CartoDB Positron as the tile layer, which loads map tiles from the internet. On Streamlit Cloud this works correctly because the server has internet access. On local machines without internet the tile layer shows as a blank grey background.

**Status:** Working correctly on deployment. No fix required.  
The circle markers and popup labels load from county coordinates hardcoded in the app and do not require internet beyond the tile layer.

---
## **Section 6: Verifying the Live Dashboard**

After deployment, verify all three tabs load and display correctly.

### **Tab 1: County Tier Map**

| What to check | Expected result |
|---------------|-----------------|
| Tier metric cards at the top | Shows count of Critical / High / Moderate / Low counties |
| Ranked bar chart | 47 counties sorted by Care Gap Index, coloured by tier |
| National average line | Red dashed vertical line on the bar chart |
| Folium map | Circle markers on Kenya map, coloured by tier, with popups |
| County data table | Full 47-county table with tier, CGI, IIT rate, VLS rate |

### **Tab 2: Dropout Risk Factors**

| What to check | Expected result |
|---------------|-----------------|
| Forest plot | Horizontal dot-and-CI-bar chart, OR=1 reference line, coloured by direction |
| Wide CI note | ever_tested_hiv bar capped at 40, instability note visible |
| Performance metrics | AUC-ROC, Recall, Precision, F1 displayed |
| Demographic Profile Explorer | Six dropdowns (County, Wealth, Distance, Age, Education, Marital) |
| Model warning (if pkl missing) | Yellow warning box — resolve by committing pkl file (see Section 5) |

### **Tab 3: Scenario Projections**

| What to check | Expected result |
|---------------|-----------------|
| Four tier line charts | BAU (blue dashed) vs Bridged Gap (red solid) IIT rate 2025–2030 |
| National headline chart | Kenya-wide IIT and VLS projections both scenarios |
| UNAIDS 95% target line | Green dotted line on VLS chart at y=0.95 |
| Patients retained | Total additional patients retained + per-tier breakdown table |
| County comparison table | Highest risk (largest CGI) and best performing (smallest CGI) |
| CSV download button | Downloads all projection data as a single CSV |
| Retraining expander | Instructions for `trigger_predictions.py --new-data-year` |

---
## **Section 7: How to Redeploy from Scratch**

Use this section if you need to start completely fresh - for example, if the repository is moved, the team changes, or the app needs to be rebuilt for a new academic year.

### **Full redeploy checklist:**

```
[ ] 1. Clone the GitHub repository to your local machine
        git clone https://github.com/<username>/<repo-name>.git

[ ] 2. Create and activate the conda environment
        conda create -n learn-env python=3.8.5
        conda activate learn-env

[ ] 3. Install all dependencies
        pip install -r requirements.txt

[ ] 4. Download raw data files and place in data/raw/
        - Adult_on_ART.xlsx         from analytics.nsdcc.go.ke/estimates
        - Adult_on_HTS.xlsx         from analytics.nsdcc.go.ke/estimates
        - HTS_Positive.xlsx         from analytics.nsdcc.go.ke/estimates
        - VLT.xlsx                  from analytics.nsdcc.go.ke/estimates
        - IIT.xlsx                  from analytics.nsdcc.go.ke/estimates
        - individual_features.csv   from dhsprogram.com (DHS Kenya 2022)

[ ] 5. Validate raw files are present
        python main.py --dry-run

[ ] 6. Run the full pipeline
        python main.py
        (takes ~5 minutes — bootstrap in Model 2 runs 500 iterations)

[ ] 7. Run trigger_alerts.py to generate iit_alerts.csv
        python app/trigger_alerts.py

[ ] 8. Verify all outputs exist
        Run Notebook 09 Section 2 (pre-flight check) to confirm

[ ] 9. Commit all processed files and model bundles
        git add data/processed/ models/
        git commit -m "Regenerated pipeline outputs for deployment"
        git push origin main

[ ] 10. Deploy to Streamlit Cloud (see Section 4)
         Main file: app/streamlit_app.py
         Branch: main

[ ] 11. Rename app URL to hivcaregapai (or desired name)

[ ] 12. Verify all three tabs load correctly (see Section 6)
```

### **Local dashboard run (without deploying)**

To run the dashboard on your local machine:
```bash
streamlit run app/streamlit_app.py --server.port 8501
```
Then open: http://localhost:8501

---
## **Section 8: How to Update the System When New Data Arrives**

NSDCC publishes updated programme data each year.  
When 2026 data (or any future year) is released, use the following process  
to update all three models and redeploy the dashboard.

### **Retraining pipeline**

A single command handles the full retraining sequence:

```bash
python app/trigger_predictions.py --new-data-year 2026
```

This script (`app/trigger_predictions.py`) runs automatically:

| Step | What happens |
|------|--------------|
| 1 | Re-cleans all 5 NSDCC files |
| 2 | Re-merges into `nsdcc_clean.csv` |
| 3 | Re-runs feature engineering → new `county_profiles.csv` |
| 4 | Retrains KMeans → new tier assignments for all 47 counties |
| 5 | Retrains Logistic Regression → updated odds ratios + 95% CI |
| 6 | Regenerates scenario projections → new forecast CSVs |
| 7 | Logs which counties changed tier → `data/processed/tier_change_log.json` |

### **After retraining**

```bash
# Commit all regenerated outputs
git add data/processed/ models/
git commit -m "Updated pipeline outputs with 2026 NSDCC data"
git push origin main
```

Streamlit Cloud will automatically redeploy with the new data.

### **Reviewing tier changes**

After retraining, check which counties moved between tiers:
```bash
cat data/processed/tier_change_log.json
```
The log shows each county's previous tier, new tier, and direction (IMPROVED or WORSENED).

### **If DHS data is also updated**

If a new DHS survey is released (e.g. Kenya DHS 2027), update Model 2 separately:

```bash
# Replace the raw DHS file
cp new_individual_features.csv data/raw/individual_features.csv

# Re-clean and retrain Model 2
python main.py --step clean_dhs
python main.py --step model2

# Commit and push
git add data/processed/individual_features_clean.csv
git add data/processed/odds_ratios_with_ci.json
git add data/processed/logreg_baseline.json
git add data/processed/dropout_risk_factors.csv
git add models/xgboost_dropout.pkl
git commit -m "Updated Model 2 with new DHS data"
git push origin main
```

---
## **Section 9: Deployment Summary and Handover Notes**

### **System overview**

| Item | Detail |
|------|--------|
| **Dashboard URL** | https://hivcaregapai.streamlit.app/ |
| **Repository** | GitHub — `main` branch |
| **Platform** | Streamlit Cloud (free tier) |
| **Auto-redeploy** | Yes — on every push to `main` |
| **Pipeline entry point** | `python main.py` |
| **Retraining entry point** | `python app/trigger_predictions.py --new-data-year <YEAR>` |
| **Local launch** | `streamlit run app/streamlit_app.py --server.port 8501` |

### **Model summary**

| Model | Algorithm | Output | File |
|-------|-----------|--------|------|
| Model 1 | KMeans (k=4) | 47 counties assigned to Critical / High / Moderate / Low tiers | `models/kmeans_county_tiers.pkl` |
| Model 2 | Logistic Regression (balanced, liblinear) | Odds ratios with 95% CI - risk factor identification | `models/xgboost_dropout.pkl` |
| Model 3 | Scenario projection | IIT + VLS rates 2025–2030, BAU vs Bridged Gap | `models/model3_scenario.pkl` |

### **Key methodological decisions documented here**

| Decision | Reason |
|----------|---------|
| Prophet replaced by scenario projection | Only 2025 data available - Prophet needs 3–4 years minimum |
| XGBoost replaced by Logistic Regression | 26 dropout cases out of 32,156 - extreme class imbalance |
| Model 2 framed as risk factor identification | 26 cases is insufficient for reliable predictive modelling |
| IIT expanded from 9 regions to 47 counties | Source data is region-level only - equal-share distribution applied |
| k=4 used despite k=3 having higher silhouette | Project requires four named intervention tiers |
| VLS improvement: ΔVLS = −0.5 × ΔIIT | Based on PEPFAR retention programme evidence |
| pkl files committed to GitHub | Required for Streamlit Cloud - no pipeline runtime available |

### **Known limitations**

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| Single-year baseline (2025 only) | No trend detection, YoY features set to 0 | Documented; scenario projection used instead of forecasting |
| IIT at region level (9 regions) | Counties in same region share IIT rate | Documented; equal-share expansion applied |
| 26 dropout cases in Model 2 | Results are descriptive, not predictive | Reframed as odds ratio analysis throughout |
| `ever_tested_hiv` CI [0.038, 6500] | Bootstrap instability for this feature | Capped at OR=40 in dashboard; noted in presentation |

### **Data sources**

| Dataset | Source | Access |
|---------|--------|---------|
| NSDCC 2025 Raw Programme Data (5 files) | analytics.nsdcc.go.ke/estimates | Public, no login |
| Kenya DHS 2022 Individual Recode | dhsprogram.com | Approved access required |

---